# 09 · Model channels, bank statements and budgets

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("channels", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


One logical storefront event may have several Kafka delivery records. Duplicate identical event payloads share one event. Conflicting payloads are recorded as exceptions. POS receipts, items and tenders keep their own grains.

The bank preview matches within legal entity by reference, currency and amount. Its results are review candidates. The source statement lacks an account identifier, so verified account evidence belongs in configuration before assigning the account. Budget comparisons must use the matching planning grain, product category for Parent and local account for Dealer.

In [2]:
display(frame(model, "order_event"), frame(model, "event_delivery"))
display(frame(model, "pos_receipt"), frame(model, "pos_receipt_line"), frame(model, "pos_tender"))

,id,legal_entity_id,source_id,sales_order_id,order_reference,status_code,event_time
0,"[""parent"",""P-E1""]",parent,P-E1,"[""parent"",""SAL-ORD-2026-00001""]",P-S1,accepted,2026-11-21T11:26:42.026905+00:00
1,"[""parent"",""P-BRONZE-R2-EVENT""]",parent,P-BRONZE-R2-EVENT,"[""parent"",""SAL-ORD-2026-00006""]",P-BRONZE-R2-ORDER,accepted,2026-11-21T15:25:39.819020+00:00
2,"[""dealer"",""D-BRONZE-R2-EVENT""]",dealer,D-BRONZE-R2-EVENT,"[""dealer"",""D-BRONZE-R2-ORDER""]",D-BRONZE-R2-ORDER,accepted,2026-11-15 03:30:00+00:00
3,"[""dealer"",""D-E1""]",dealer,D-E1,"[""dealer"",""D-S1""]",D-S1,accepted,2026-11-05 03:30:00+00:00


,id,bronze_key,topic,generation,partition,offset,ingested_at,event
0,kafka/parent.orders.v1/generation=v1/partition...,kafka/parent.orders.v1/generation=v1/partition...,parent.orders.v1,v1,0,0,2026-11-21T13:02:46.761294+00:00,"{'lines': [{'sku': 'P-PUMP', 'quantity': 10, '..."
1,kafka/parent.orders.v1/generation=v1/partition...,kafka/parent.orders.v1/generation=v1/partition...,parent.orders.v1,v1,0,1,2026-11-21T15:27:38.889265+00:00,"{'lines': [{'sku': 'P-PUMP', 'quantity': 2, 'u..."


,id,legal_entity_id,source_id,branch,sold_at,sales_order_id
0,"[""parent"",""P-BRONZE-R2-POS""]",parent,P-BRONZE-R2-POS,parent-branch,None,None
1,"[""parent"",""P-POS1""]",parent,P-POS1,parent-branch,None,None
2,"[""dealer"",""D-POS1""]",dealer,D-POS1,dealer-branch,2026-11-05 04:30:00+00:00,None


,id,pos_receipt_id,source_product_id,quantity,unit_price,amount,amount_basis,line_identity_basis
0,"[""parent"",""P-BRONZE-R2-POS"",1]","[""parent"",""P-BRONZE-R2-POS""]","[""parent"",""P-PUMP""]",1,10000,10000,quantity times price,payload position
1,"[""parent"",""P-POS1"",1]","[""parent"",""P-POS1""]","[""parent"",""P-PUMP""]",1,10000,10000,quantity times price,payload position
2,"[""dealer"",""D-PL1""]","[""dealer"",""D-POS1""]","[""dealer"",""D-PUMP""]",1,NaN,10000,recorded,NaN


,id,pos_receipt_id,method,amount
0,"[""parent"",""P-BRONZE-R2-POS"",""tender""]","[""parent"",""P-BRONZE-R2-POS""]",card,None
1,"[""parent"",""P-POS1"",""tender""]","[""parent"",""P-POS1""]",card,None
2,"[""dealer"",""D-PT1""]","[""dealer"",""D-POS1""]",card,10000


In [3]:
display(frame(model, "bank_statement_line"), frame(model, "bank_match_candidate"))
display(frame(model, "budget_amount"))
display(pd.DataFrame(model.exceptions))

,id,legal_entity_id,bank_account_id,posting_date,date_basis,source_id,payment_reference,amount,currency
0,"[""parent"",""P-B1""]",parent,None,2026-11-10,statement date,P-B1,P-P1,25000,INR
1,"[""dealer"",""D-B1""]",dealer,None,2026-11-12,booking day,D-B1,D-P1,10000.00,INR


,id,bank_statement_line_id,payment_id,status,evidence
0,"[""parent"",""P-B1""]","[""parent"",""P-B1""]","[""parent"",""ACC-PAY-2026-00001""]",review,"same legal entity, payment reference, currency..."
1,"[""dealer"",""D-B1""]","[""dealer"",""D-B1""]","[""dealer"",""D-R1""]",review,"same legal entity, payment reference, currency..."


,id,legal_entity_id,period,currency,scope,scope_type,scenario,input_version,amount
0,"[""parent"",""product_category"",""PUMP"",""2026-11"",...",parent,2026-11,INR,PUMP,product_category,plan,files/otc-inbound/parent/finance/dealer-order-...,120000
1,"[""dealer"",""local_account"",""400-D"",""2026-11"",""I...",dealer,2026-11,INR,400-D,local_account,plan,files/otc-inbound/dealer/finance/dealer-order-...,50000.00


,rule,table,id,severity,detail
0,unresolved_reference,billing_document,"[""dealer"",""D-BRONZE-R2-CREDIT""]",review,"{'role': 'original_invoice', 'target': 'billin..."
1,bank_account_not_in_feed,bank_statement_line,"[""parent"",""P-B1""]",review,P-P1
2,bank_account_not_in_feed,bank_statement_line,"[""dealer"",""D-B1""]",review,D-P1
